# PathSnap — Member 1 EDA
Train and student-test only. Read the measured inventory, quality, alignment and candidate-sweep outputs. AI-assisted notebook; all values come from executed local analysis. See `report/member1/README.md` for reproduction and uncertainty.

In [1]:
def display(value):
    print(value.to_string() if hasattr(value, 'to_string') else value)

from pathlib import Path
import os
if Path.cwd().name == 'notebooks':
    os.chdir('..')
import pandas as pd
quality = pd.read_csv('results/metrics/quality_report.csv', dtype={'trip_id': str})
display(quality.groupby('partition')[['n_obs','n_nodes','n_arcs','route_breaks','missing_values','large_gaps']].agg(['sum','min','median','max']))

               n_obs                      n_nodes                         n_arcs                        route_breaks                missing_values                large_gaps                
                 sum  min  median    max      sum   min  median     max      sum   min   median     max          sum min median max            sum min median max        sum min median  max
partition                                                                                                                                                                                   
student_test   19160  262  1420.0   3958    90689  1770  6596.5   26635   185041  3577  13145.0   56027            1   0    0.0   1              0   0    0.0   0         44   0    0.0   39
train         208417  361  1529.0  19313  1933485   299  9608.0  155003  3876904   649  19526.5  307392           13   0    0.0   3              0   0    0.0   0       1216   0    1.0  320


In [2]:
inventory = pd.read_csv('results/metrics/inventory.csv', dtype={'trip_id': str})
display(inventory.groupby(['partition','file_type']).agg(files=('trip_id','count'), rows=('rows','sum'), bytes=('bytes','sum')))
display(quality.sort_values('max_gap_s', ascending=False).head(10))

                        files     rows     bytes
partition    file_type                          
student_test arcs          10   185041   1902157
             nodes         10    90689   1822179
             route         10     3477     18344
             track         10    19160    625911
train        arcs          80  3876904  44267751
             nodes         80  1933485  38431647
             route         80    31336    165702
             track         80   208417   6872352
   partition   trip_id  n_obs  n_nodes  n_arcs  n_route_arcs  missing_values  invalid_coordinates  duplicate_timestamps  reversed_timestamps  irregular_gaps  large_gaps  max_gap_s     lon_min     lon_max    lat_min    lat_max  weak_components  strong_components  isolated_nodes  zero_length_arcs  reverse_arc_share  route_breaks                                                                   metric_crs
72     train  00000090    536     4010    7942           235               0                    0       

## Approximate labels
Offline dynamic programming aligns GPS observations with ordered reference arc occurrences. Assignments beyond 100 m are unknown, never negative training examples. Compare with independently measured nearest-reference distances; inspect offset trips before interpreting pointwise accuracy.

In [3]:
alignment = pd.read_csv('results/metrics/label_alignment.csv', dtype={'trip_id': str})
display(alignment.groupby('partition')[['n_obs','unknown_labels','route_breaks']].sum())
display(alignment.sort_values('unknown_labels', ascending=False).head(10))
display(pd.read_csv('results/metrics/reference_distance_diagnostics.csv', dtype={'trip_id': str}).sort_values('nearest_reference_over100m', ascending=False).head(10))

               n_obs  unknown_labels  route_breaks
partition                                         
student_test   19160              17             1
train         208417           20540            13
       partition   trip_id  n_obs  unknown_labels  median_distance_m  p95_distance_m  max_distance_m  skipped_route_occurrences  route_breaks
44         train  00000054  19313           16455         140.116231      210.093911      241.309417                         34             0
26         train  00000033   5844            2978         122.421322      142.068025      160.472081                         32             1
1          train  00000002   7243             719           5.794087      242.387099      569.727677                          1             1
53         train  00000066   4325             198           6.591532       65.540984      647.260707                          2             0
64         train  00000081   3224              63           4.567061       44.566253  

## Training-only candidate sweep
Search radius/K are selected on training observations. Known-label candidate miss rate sets the pointwise Top-1 ceiling; excluded unknown observations are reported separately. Scaling is fitted inside training folds, never during EDA.

In [4]:
display(pd.read_csv('results/metrics/candidate_sweep.csv'))


   radius_m   k  known_obs  misses  n_candidates   n_obs  candidate_miss_rate  mean_candidates
0        25   4     187877    8904        630511  208417             0.047393         3.025238
1        25   8     187877    4909        834400  208417             0.026129         4.003512
2        25  16     187877    4265        938493  208417             0.022701         4.502958
3        50   4     187877    7894        716436  208417             0.042017         3.437512
4        50   8     187877    3402       1124857  208417             0.018108         5.397146
5        50  16     187877    2548       1551906  208417             0.013562         7.446158
6       100   4     187877    7663        769597  208417             0.040787         3.692583
7       100   8     187877    2797       1361726  208417             0.014887         6.533661
8       100  16     187877    1793       2263447  208417             0.009543        10.860184


### Measured plots
![eda_timestamp_gaps](../results/plots/eda_timestamp_gaps.png)

![eda_n_obs](../results/plots/eda_n_obs.png)

![eda_n_arcs](../results/plots/eda_n_arcs.png)

![eda_missing_values](../results/plots/eda_missing_values.png)

![eda_connectivity](../results/plots/eda_connectivity.png)

![eda_trajectory](../results/plots/eda_trajectory.png)

![eda_candidate_sweep](../results/plots/eda_candidate_sweep.png)

![Development geography](../results/plots/eda_geography.png)

![Reference-distance outlier](../results/plots/eda_reference_offset.png)


## Interactive train map
Open [the train GPS/network map](../maps/eda_train_00000001.html). It has separate road/GPS layers and a legend. Predictions and reference-route comparison maps are Member 3’s deliverable.

In [5]:
# Optional full regeneration, from the repository root:
# import subprocess, sys
# subprocess.run([sys.executable, '-m', 'src.data_processing.run_member1'], check=True)